
# TRNG Approach A - RTC vs TCC Jitter on XMEGA (CW303)
Quick testbench to compile, flash, collect, and evaluate random bits from a
TRUERA-style TRNG adapted to the ATxmega128D4 on ChipWhisperer CW303.

> Duplicate this notebook to create Approach B/C/D (e.g., ADC noise, ring oscillators, SRAM startup), changing only the acquisition cell title & acquisition code. Each notebook keeps the same test battery, so you can compare results apples-to-apples.



## Requirements
- ChipWhisperer installed (`pip install chipwhisperer`), hardware connected (CW-Lite/NAE-Scope + CW303 XMEGA target).
- Your TRNG firmware placed in `FW_DIR` (below) and builds with `make PLATFORM=CW303` to create `.hex`.
- The firmware implements SimpleSerial command: `r<N>` -> returns `ceil(N/8)` bytes (MSB-first packing).

If you use my minimal no-FIFO firmware from chat, you're good.


## 1) Build firmware

In [246]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange

scope = cw.scope()
target = cw.target(scope, cw.targets.SimpleSerial2)
scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 12208994                  to 35137417                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0               
scope.clock.clkgen_

In [ ]:
%%bash
make PLATFORM=CWLITEXMEGA SOURCE=osc_trng.c  SS_VER=SS_VER_2_1

## 2) Program target (XMEGA on CW303)

In [237]:
hex = "output-CWLITEXMEGA.hex"
cw.program_target(scope, cw.programmers.XMEGAProgrammer, hex)

XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 4431 bytes


## 3) Acquire random bits - Approach A: RTC vs TCC jitter

In [247]:
target.flush()
target.simpleserial_write('c', b'')
data = target.simpleserial_read('r', 3)
print(data)

CWbytearray(b'a5 5a c0')


In [ ]:
# Request 16 random bytes
target.flush()
N = 260
payload = N.to_bytes(2, 'little')  # 1-byte request
target.simpleserial_write('b', payload)
# Read N bytes back
buf = bytearray()
while len(buf) < N:
    want = min(249, N - len(buf))
    chunk = target.simpleserial_read('r', want, timeout=50000000)
    if chunk is None:
        raise RuntimeError("Timeout waiting for target")
    buf.extend(chunk)

print(buf)

(ChipWhisperer Target WARNING|File SimpleSerial2.py:506) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:287) Device did not ack


bytearray(b'\xe2(\xd8\xa1\xde\x0b\xb6\xb1\xd9\x15fW\x98\x93\xed\xa7\x01\xed\xf7\xb4\xaf\x18Y\x13\xaf\x95Jc\xaa"\x08\x9f\xebn\xb4\x04\xaf\x82\xd2\xa5\xbe\xa0\xa0\xa9\xa4\xb3\xf9&\x08|J\xbe\x91\xe0\xe5\xb8\x03X\x03\x0c\xb4KP\xb5\xac\x7f\x84\xb69\xca\xde\xe8\xb0=i\xf8@\x9a\x0e\'|\xa4\xfe\xbc&\xc1\xcc\xe0SX\x85\xd6:0\xd6\t@hc\xc8\xa57"J\x9f\x9fO1ss\xae\xd1\xbdS\xb8&0\x90@\x80?\x13\xae\x86\xef\xf5\xc8Yr\xbd\xa9NI\xfe+\xc2w\r5\x85\xf3\xfd\xc6\xf8\xbc4\x85\xd1Y{\xef\xa9\x03\xa4\xc7\'\xbf_\xa5U\xff\xb4\x96\xcdJ\xbd0\xae\xb4^\xf0w\xa9\xf4\xcc\x86h\xf4miZ\x18\xfa\xc8\xd7\xbe\xcdn\xe7\n\x91\x94q\x16\xf3\xb2\xdf\xdfR\xbc\x89\xcaR\xe3X\xb6\xa7\x01\xe3\xfdb\xefg?5\xa4m\xe0\x1b\xf3{HR\x12\xe5\xc7\xfeA\xf4N\xdf\x99X\xd5\x08\xb2\x08\x80&\xd0D\x03rqg\xcc\xf8Mq\xf59\x0c\xb2\x03\xea\x08\xeeS\xeeD')


In [ ]:
import numpy as np

N = 1000

# Flush old data (optional)
try:
    target.flush()
except Exception:
    pass

# ---------- Request ----------
payload = N.to_bytes(2, 'little')   # fits in 2 bytes (<= 65535)
target.simpleserial_write('b', payload)

# Read N bytes back
buf = bytearray()
while len(buf) < N:
    want = min(249, N - len(buf))
    chunk = target.simpleserial_read('r', want, timeout=5000000)
    if chunk is None:
        raise RuntimeError("Timeout waiting for target")
    buf.extend(chunk)

print(f"Received {len(buf)} bytes")

# ---------- Analysis ----------
data = np.frombuffer(buf, dtype=np.uint8)
biases = []

for b in range(8):
    ones = np.count_nonzero((data >> b) & 1)
    p1 = ones / len(data)
    biases.append(p1)
    print(f"bit {b}: p1={p1:.5f}")

# ---------- Plot ----------
plt.figure(figsize=(7,4))
plt.bar(range(8), biases, color="royalblue")
plt.axhline(0.5, color="red", linestyle="--", label="ideal = 0.5")
plt.xticks(range(8), [f"bit {i}" for i in range(8)])
plt.ylim(0,1)
plt.ylabel("Probability of 1")
plt.title("Bit bias per position (from 10,000 bytes)")
plt.legend()
plt.show()


In [ ]:
target.flush()
target.simpleserial_write('u', b'')
resp = target.simpleserial_read('d', 4)
print("Resp:", resp.hex() if resp else None)


In [ ]:
target.simpleserial_write('x', b'')
resp = target.simpleserial_read('x', 2)
print("Resp:", resp.hex() if resp else None)

In [ ]:

target.simpleserial_write('o', b'')
resp = target.simpleserial_read('x', 2)
print("Resp:", resp.hex() if resp else None)


In [ ]:
target.simpleserial_write('a', b'')
resp = target.simpleserial_read('x', 2,timeout=2000)  
b0 = resp[0]   # first byte (0xA1)
b1 = resp[1]   # second byte (0x97)
print(f"{b0:02x}", f"{b1:02x}")  # prints just hex digits, always 2 chars
print("Resp:", resp.hex() if resp else None)

In [ ]:
f_rtc = 32.768e3
t_rtc = 1 / f_rtc
f_tcc0 = 32e6
t_tcc0 = 1 / f_tcc0

print(log2(1*t_rtc / t_tcc0))

## senity check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 1. Collect random bytes from target =====
N = 1000   # number of random bytes to collect
rand_bytes = []

for _ in range(N):
    target.simpleserial_write('a', b'')              # request one byte
    resp_lsb = target.simpleserial_read('x', 2, timeout=2000)
    if resp_lsb:
        resp_first_byte = resp_lsb[0]
        resp_second_byte = resp_lsb[1]
        rand_bytes.append(resp_first_byte)
        # rand_bytes.append(resp_second_byte)
        


rand_bytes = np.array(rand_bytes, dtype=np.uint8)

print(f"Collected {len(rand_bytes)} random bytes")

# ===== 2. Bit bias analysis =====
bit_counts = np.zeros(8)
for i in range(8):
    bit_counts[i] = np.sum((rand_bytes >> i) & 1)

bit_bias = bit_counts / len(rand_bytes)  # fraction of 1s per bit

print("Bit biases (probability of '1' for each bit position):")
for i, p in enumerate(bit_bias):
    print(f"Bit {i}: {p:.4f}")

plt.bar(range(8), bit_bias)
plt.axhline(0.5, color='r', linestyle='--')
plt.xlabel("Bit position (0 = LSB)")
plt.ylabel("Probability of '1'")
plt.title("Bit Bias")
plt.show()

# ===== 3. Byte value distribution =====
values, counts = np.unique(rand_bytes, return_counts=True)
probs = counts / len(rand_bytes)

print("\nByte distribution stats:")
print(f"Min prob: {probs.min():.4f}, Max prob: {probs.max():.4f}, Expected = {1/256:.4f}")

plt.hist(rand_bytes, bins=256, range=(0, 255), density=True, color='skyblue')
plt.axhline(1/256, color='r', linestyle='--', label="Uniform expected")
plt.xlabel("Byte value (0–255)")
plt.ylabel("Probability")
plt.title("Distribution of TRNG Byte Values")
plt.legend()
plt.show()


## 4) Quick tests & metrics

## 5) Save a one-page text report (quick summary)

In [ ]:

report = f"""TRNG Approach A - RTC vs TCC jitter (XMEGA CW303)
Bits collected: {bits.size}

Frequency (monobit):
  ones = {freq['ones']}, p_hat = {freq['p_hat']:.6f}, z = {freq['z']:.3f}

Runs test:
  runs = {runs.get('runs')}, expected = {runs.get('expected', float('nan')):.2f},
  z = {runs.get('z', float('nan')):.3f}, p = {runs.get('p_value', float('nan')):.4f}

Entropy:
  Shannon entropy/bit (via bytes) = {Hperbit:.6f}
  Min-entropy/bit (bit MCV)       = {Hmin:.6f}

Notes:
  - This is a quick sanity battery; not a replacement for full NIST STS.
  - Duplicate this notebook for other approaches (ADC noise, RO, SRAM) and re-run.
"""

path = f"/mnt/data/{SAVE_PREFIX}_summary.txt"
with open(path, "w") as f:
    f.write(report)
print(report)
print("Saved report:", path)


## 6) Cleanup / disconnect (optional)

In [245]:

try:
    target.dis()
except Exception:
    pass
try:
    scope.dis()
except Exception:
    pass
print("Disconnected.")


Disconnected.
